# Independent Fold 3 methodological audit

## tl;dr

The committed Fold 3 arithmetic, locked-gate decisions, pooled raw aggregation, equal-volume construction, and temporal outcome labels reconcile. RB carry legitimately passes the frozen point gates and may advance unchanged; RB opportunity must remain shadow because the pre-existing direction rule fails on 2021 decreases. This is a fragile point-gate pass—not validation. Carry's 2023 lift interval crosses zero, carry-only alerts are weak relative to alerts overlapping RB opportunity, and the source manifest overstates literal post-2023 file isolation even though no 2024–2025 values entered scoring.

## Context & Methods

Audited commit: `a18c5cc3e8c9124be4781bececea0a93f7b4faf8`. Sources are the committed Fold 1/2/3 alert archives, Fold 3 enriched canonical archive, locked protocol/configuration, and independently generated audit CSVs.

### Key Assumptions

- Primary policy excludes confirmed partial games and retains suspected cases.
- Precision uses only rows with a two-game persistence label; reversion uses rows with a next-game label.
- Precision CI reproduces the locked 2,000-row bootstrap with seed 850. Lift CI reproduces the 2,000 season-week cluster bootstrap.
- This notebook does not select alerts, tune rules, execute Fold 4, or read any 2024–2025 result rows.

In [1]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
F3 = ROOT/'outputs'/'role_validation'/'fold_3'
AUDIT = ROOT/'outputs'/'role_validation'/'fold_3_independent_audit'
PRIMARY = 'PRIMARY_CONFIRMED_EXCLUDED'
RB = ['rb_carry_share','rb_opportunity_share']
alerts = pd.read_csv(F3/'fold3_alerts_2023.csv.gz', low_memory=False)
canonical = pd.read_csv(F3/'canonical_role_2023_enriched.csv.gz', low_memory=False)
primary = alerts.query('partial_policy == @PRIMARY and role_family in @RB').copy()
assert set(alerts.season.unique()) == {2023}
assert set(canonical.season.unique()) == {2023}
len(alerts), len(canonical)

(1956, 7448)

## Data

### 1. Confirm grain, seasons, and archive identity

In [2]:
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()

key=['partial_policy','role_family','method','season','week','player_id','team']
profile={
 'alert_rows':len(alerts), 'canonical_rows':len(canonical),
 'duplicate_alert_keys':int(alerts.duplicated(key,keep=False).sum()),
 'alert_archive_sha256':sha256(F3/'fold3_alerts_2023.csv.gz'),
 'config_sha256':sha256(ROOT/'config'/'role_change_fold2_candidate.yaml'),
}
profile

{'alert_rows': 1956,
 'canonical_rows': 7448,
 'duplicate_alert_keys': 0,
 'alert_archive_sha256': 'f9c4b971c0abb15662f66f3cc2d9103df0bed257b718408c82543f4f0cc943ee',
 'config_sha256': '4dcf389a1f8fcdd11a9277305a8372fadaabaa830185e07eff5d8fbb274a81c7'}

## Results

### 2. Independently recompute headline numerators and rates

In [3]:
def ind(s):
    if pd.api.types.is_bool_dtype(s): return s.astype(float)
    return s.map({True:1.,False:0.,'True':1.,'False':0.})

def summarize(g):
    p=ind(g.persistent); r=ind(g.immediate_reversion); ev=p.notna()
    return pd.Series({
      'alerts':len(g),'evaluable':int(ev.sum()),'persistent':int(p.sum()),
      'precision':p.mean(),'reversion_n':int(r.notna().sum()),
      'reversions':int(r.sum()),'reversion':r.mean(),
      'median_retention':pd.to_numeric(g.loc[ev,'retention']).median()})

headline=(primary.groupby(['role_family','method'],sort=True).apply(summarize,include_groups=False).reset_index())
headline

,role_family,method,alerts,evaluable,persistent,precision,reversion_n,reversions,reversion,median_retention
0,rb_carry_share,full_propwar,60.0,47.0,31.0,0.659574,52.0,12.0,0.230769,0.794931
1,rb_carry_share,naive_spike,60.0,49.0,26.0,0.530612,52.0,17.0,0.326923,0.526657
2,rb_carry_share,normal_game_trend,60.0,49.0,31.0,0.632653,53.0,11.0,0.207547,0.754819
3,rb_carry_share,two_week_raw,60.0,49.0,33.0,0.673469,53.0,11.0,0.207547,0.745771
4,rb_opportunity_share,full_propwar,74.0,57.0,44.0,0.771930,64.0,10.0,0.156250,0.910085
5,rb_opportunity_share,naive_spike,74.0,58.0,33.0,0.568966,63.0,20.0,0.317460,0.616665
6,rb_opportunity_share,normal_game_trend,74.0,58.0,42.0,0.724138,64.0,10.0,0.156250,0.730123
7,rb_opportunity_share,two_week_raw,74.0,59.0,39.0,0.661017,64.0,12.0,0.187500,0.702360


### 3. Verify bootstrap intervals and full-versus-naive comparisons

In [4]:
def rate_ci(s, iterations=2000, seed=850):
    x=ind(s).dropna().to_numpy(); rng=np.random.default_rng(seed)
    samples=rng.choice(x,size=(iterations,len(x)),replace=True).mean(1)
    return np.quantile(samples,[.025,.975])

ci=[]
for (family,method),g in primary.groupby(['role_family','method']):
    lo,hi=rate_ci(g.persistent); ci.append((family,method,lo,hi))
ci=pd.DataFrame(ci,columns=['role_family','method','ci_low','ci_high'])
comparison=pd.read_csv(AUDIT/'family_comparisons_recomputed_2023.csv')
display(ci,comparison)

,role_family,method,ci_low,ci_high
0,rb_carry_share,full_propwar,0.531915,0.787234
1,rb_carry_share,naive_spike,0.387755,0.673469
2,rb_carry_share,normal_game_trend,0.509694,0.755102
3,rb_carry_share,two_week_raw,0.551020,0.795918
4,rb_opportunity_share,full_propwar,0.666667,0.877193
5,rb_opportunity_share,naive_spike,0.448276,0.689655
6,rb_opportunity_share,normal_game_trend,0.603448,0.844828
7,rb_opportunity_share,two_week_raw,0.542373,0.779661


,period,role_family,full_alerts,full_evaluable_alerts,full_persistent_alerts,full_precision,naive_alerts,naive_evaluable_alerts,naive_persistent_alerts,naive_precision,...,precision_improvement_ci_high,full_reversion_evaluable_alerts,full_immediate_reversions,full_reversion_rate,naive_reversion_evaluable_alerts,naive_immediate_reversions,naive_reversion_rate,reversion_improvement,full_median_retention,naive_median_retention
0,untouched_2023,rb_carry_share,60,47,31,0.659574,60,49,26,0.530612,...,0.263314,52,12,0.230769,52,17,0.326923,0.096154,0.794931,0.526657
1,untouched_2023,rb_opportunity_share,74,57,44,0.771930,74,58,33,0.568966,...,0.344047,64,10,0.156250,63,20,0.317460,0.161210,0.910085,0.616665


### 4. Prove pooled 2022–2023 results use raw numerators and denominators

In [5]:
pooled=pd.read_csv(AUDIT/'pooled_recomputed_2022_2023.csv')
proof=pooled[['role_family','full_persistent_alerts','full_evaluable_alerts','full_precision',
 'naive_persistent_alerts','naive_evaluable_alerts','naive_precision','precision_improvement',
 'full_immediate_reversions','full_reversion_evaluable_alerts','full_reversion_rate','full_median_retention']]
assert np.allclose(proof.full_persistent_alerts/proof.full_evaluable_alerts,proof.full_precision)
assert np.allclose(proof.naive_persistent_alerts/proof.naive_evaluable_alerts,proof.naive_precision)
proof

,role_family,full_persistent_alerts,full_evaluable_alerts,full_precision,naive_persistent_alerts,naive_evaluable_alerts,naive_precision,precision_improvement,full_immediate_reversions,full_reversion_evaluable_alerts,full_reversion_rate,full_median_retention
0,rb_carry_share,56,86,0.651163,44,87,0.505747,0.145416,18,92,0.195652,0.738639
1,rb_opportunity_share,73,104,0.701923,57,103,0.553398,0.148525,17,114,0.149123,0.793768


### 5. Verify every locked gate and the 2021 opportunity decrease cell

In [6]:
gates=pd.read_csv(AUDIT/'gate_by_gate_independent_check.csv')
directions=pd.read_csv(AUDIT/'cross_season_direction_recomputed_2021_2023.csv')
failing=directions.query("role_family == 'rb_opportunity_share' and period == 'redeveloped_2021' and direction == 'decrease'")
display(gates,failing)
assert len(failing)==1 and failing.iloc[0].precision_improvement < 0

,role_family,gate,observed,threshold,passed
0,rb_carry_share,min_holdout_alerts,60,>= 50,True
1,rb_carry_share,min_persistence_precision,0.6595744680851063,>= 0.6,True
2,rb_carry_share,min_absolute_improvement_vs_naive,0.12896222318714712,>= 0.1,True
3,rb_carry_share,max_immediate_reversion_rate,0.23076923076923078,<= 0.25,True
4,rb_carry_share,min_reversion_improvement_vs_naive,0.09615384615384615,>= 0.08,True
5,rb_carry_share,min_median_retention,0.7949311152875244,>= 0.5,True
6,rb_carry_share,min_alerts_per_week,3.3333333333333335,>= 0.5,True
7,rb_carry_share,direction_consistent_across_periods,True,all available period-direction lifts >= 0,True
8,rb_carry_share,frozen_before_holdout,True,required,True
9,rb_opportunity_share,min_holdout_alerts,74,>= 50,True


,period,role_family,direction,full_alerts,full_evaluable_alerts,full_persistent_alerts,full_precision,full_reversion_evaluable_alerts,full_immediate_reversions,full_reversion_rate,...,naive_alerts,naive_evaluable_alerts,naive_persistent_alerts,naive_precision,naive_reversion_evaluable_alerts,naive_immediate_reversions,naive_reversion_rate,naive_median_retention,precision_improvement,reversion_improvement
2,redeveloped_2021,rb_opportunity_share,decrease,33,22,14,0.636364,26,3,0.115385,...,34,23,15,0.652174,27,4,0.148148,0.802114,-0.01581,0.032764


### 6. Inspect subgroup dependence

In [7]:
subgroups=pd.read_csv(AUDIT/'subgroup_metrics_2023.csv')
overlap=pd.read_csv(AUDIT/'carry_opportunity_overlap_dependence.csv')
concentration=pd.read_csv(AUDIT/'concentration_summary_2023.csv')
partial=pd.read_csv(AUDIT/'partial_policy_sensitivity_recomputed_2023.csv')
display(subgroups.query("role_family == 'rb_carry_share'"),overlap,concentration,partial)

,role_family,dimension,subgroup,alerts,evaluable_alerts,persistent_alerts,precision,reversion_evaluable_alerts,immediate_reversions,reversion_rate,median_retention
0,rb_carry_share,direction,decrease,31,25,15,0.600000,27,7,0.259259,0.702476
1,rb_carry_share,direction,increase,29,22,16,0.727273,25,5,0.200000,0.829902
4,rb_carry_share,week_block,weeks_1_6,2,2,1,0.500000,2,0,0.000000,0.635210
5,rb_carry_share,week_block,weeks_7_12,28,28,19,0.678571,28,8,0.285714,0.755859
6,rb_carry_share,week_block,weeks_13_18,30,17,11,0.647059,22,4,0.181818,0.826709
10,rb_carry_share,partial_status_group,not_suspected,58,46,30,0.652174,51,12,0.235294,0.775915
11,rb_carry_share,partial_status_group,suspected,2,1,1,1.000000,1,0,0.000000,1.131154
14,rb_carry_share,denominator_band,18_20,15,13,8,0.615385,13,5,0.384615,0.833095
15,rb_carry_share,denominator_band,21_24,16,14,10,0.714286,15,3,0.200000,0.775915
16,rb_carry_share,denominator_band,25_29,20,14,8,0.571429,17,2,0.117647,0.761533


,period,role_family,overlap_group,alerts,evaluable_alerts,persistent_alerts,precision,reversion_evaluable_alerts,immediate_reversions,reversion_rate,median_retention
0,untouched_2022,rb_carry_share,also_opportunity,33,27,15,0.555556,27,4,0.148148,0.550887
1,untouched_2022,rb_carry_share,carry_only,16,12,10,0.833333,13,2,0.153846,0.738639
2,untouched_2023,rb_carry_share,also_opportunity,41,33,25,0.757576,35,5,0.142857,0.838510
3,untouched_2023,rb_carry_share,carry_only,19,14,6,0.428571,17,7,0.411765,0.388340
4,pooled_2022_2023,rb_carry_share,also_opportunity,74,60,40,0.666667,62,9,0.145161,0.772363
5,pooled_2022_2023,rb_carry_share,carry_only,35,26,16,0.615385,30,9,0.300000,0.630897


,role_family,dimension,alerts,unique_groups,top_1_share,top_3_share,top_5_share,hhi,effective_groups,leave_one_out_precision_min,leave_one_out_precision_max
0,rb_carry_share,team,60,24,0.100000,0.250000,0.383333,0.055556,18.000000,0.627907,0.688889
1,rb_carry_share,player,60,39,0.066667,0.166667,0.266667,0.032222,31.034483,0.644444,0.681818
2,rb_opportunity_share,team,74,28,0.094595,0.256757,0.378378,0.050402,19.840580,0.754717,0.796296
3,rb_opportunity_share,player,74,45,0.054054,0.162162,0.243243,0.029218,34.225000,0.759259,0.796296


,period,role_family,full_alerts,full_evaluable_alerts,full_persistent_alerts,full_precision,naive_alerts,naive_evaluable_alerts,naive_persistent_alerts,naive_precision,...,precision_improvement_ci_high,full_reversion_evaluable_alerts,full_immediate_reversions,full_reversion_rate,naive_reversion_evaluable_alerts,naive_immediate_reversions,naive_reversion_rate,reversion_improvement,full_median_retention,naive_median_retention
0,ALL_INCLUDED,rb_carry_share,61,47,31,0.659574,61,49,27,0.551020,...,0.241880,53,13,0.245283,53,17,0.320755,0.075472,0.754819,0.527938
1,ALL_INCLUDED,rb_opportunity_share,75,58,45,0.775862,75,58,33,0.568966,...,0.347446,65,10,0.153846,64,19,0.296875,0.143029,0.896747,0.616665
2,PRIMARY_CONFIRMED_EXCLUDED,rb_carry_share,60,47,31,0.659574,60,49,26,0.530612,...,0.263314,52,12,0.230769,52,17,0.326923,0.096154,0.794931,0.526657
3,PRIMARY_CONFIRMED_EXCLUDED,rb_opportunity_share,74,57,44,0.771930,74,58,33,0.568966,...,0.344047,64,10,0.156250,63,20,0.317460,0.161210,0.910085,0.616665
4,STRICT_SUSPECTED_EXCLUDED,rb_carry_share,59,48,31,0.645833,59,49,26,0.530612,...,0.301933,52,12,0.230769,52,17,0.326923,0.096154,0.791804,0.526657
5,STRICT_SUSPECTED_EXCLUDED,rb_opportunity_share,71,55,42,0.763636,71,57,33,0.578947,...,0.330762,63,9,0.142857,62,19,0.306452,0.163594,0.930276,0.635986


### 7. Verify equal volume, comparator quality, temporal order, and reconstructed outcomes

In [8]:
equal=pd.read_csv(AUDIT/'equal_volume_independent_check.csv')
fairness=pd.read_csv(AUDIT/'comparator_fairness_selected_rows.csv')
rule_compliance=pd.read_csv(AUDIT/'full_alert_rule_compliance.csv')
replay=pd.read_csv(AUDIT/'comparator_selection_replay.csv')
temporal=pd.read_csv(AUDIT/'temporal_integrity_independent_check.csv')
outcomes=pd.read_csv(AUDIT/'outcome_label_reconstruction.csv')
assert len(equal)==216 and equal.equal_volume.all()
assert temporal.passed.all() and outcomes.matched.all()
assert rule_compliance.all_rules_satisfied.all()
assert replay.pool_sufficient.all() and replay.selection_matches_deterministic_top_n.all()
assert (fairness[['data_quality_pass_rate','qualifying_game_rate','identity_resolved_rate']]==1).all().all()
display(fairness,rule_compliance,replay.groupby(['method'])[['pool_sufficient','selection_matches_deterministic_top_n']].all(),temporal,outcomes)

,role_family,method,alerts,unique_players,data_quality_pass_rate,qualifying_game_rate,identity_resolved_rate,feature_eligible_rate,minimum_baseline_n,confirmation_window_match_rate,outcome_evaluable_rate
0,rb_carry_share,full_propwar,60,39,1.0,1.0,1.0,1.0,4,1.0,0.783333
1,rb_carry_share,naive_spike,60,38,1.0,1.0,1.0,1.0,4,1.0,0.816667
2,rb_carry_share,normal_game_trend,60,28,1.0,1.0,1.0,1.0,4,1.0,0.816667
3,rb_carry_share,two_week_raw,60,29,1.0,1.0,1.0,1.0,4,1.0,0.816667
4,rb_opportunity_share,full_propwar,74,45,1.0,1.0,1.0,1.0,4,1.0,0.770270
5,rb_opportunity_share,naive_spike,74,44,1.0,1.0,1.0,1.0,4,1.0,0.783784
6,rb_opportunity_share,normal_game_trend,74,34,1.0,1.0,1.0,1.0,4,1.0,0.783784
7,rb_opportunity_share,two_week_raw,74,35,1.0,1.0,1.0,1.0,4,1.0,0.797297


,partial_policy,role_family,alerts,quality_violations,qualifying_violations,identity_violations,baseline_minimum_violations,confirmation_complete_violations,strict_confirmation_violations,absolute_delta_violations,player_opportunity_violations,team_denominator_violations,all_rules_satisfied
0,ALL_INCLUDED,rb_carry_share,61,0,0,0,0,0,0,0,0,0,True
1,ALL_INCLUDED,rb_opportunity_share,75,0,0,0,0,0,0,0,0,0,True
2,ALL_INCLUDED,te_target_share,4,0,0,0,0,0,0,0,0,0,True
3,ALL_INCLUDED,wr_target_share,29,0,0,0,0,0,0,0,0,0,True
4,PRIMARY_CONFIRMED_EXCLUDED,rb_carry_share,60,0,0,0,0,0,0,0,0,0,True
5,PRIMARY_CONFIRMED_EXCLUDED,rb_opportunity_share,74,0,0,0,0,0,0,0,0,0,True
6,PRIMARY_CONFIRMED_EXCLUDED,te_target_share,4,0,0,0,0,0,0,0,0,0,True
7,PRIMARY_CONFIRMED_EXCLUDED,wr_target_share,27,0,0,0,0,0,0,0,0,0,True
8,STRICT_SUSPECTED_EXCLUDED,rb_carry_share,59,0,0,0,0,0,0,0,0,0,True
9,STRICT_SUSPECTED_EXCLUDED,rb_opportunity_share,71,0,0,0,0,0,0,0,0,0,True


,pool_sufficient,selection_matches_deterministic_top_n
method,,
naive_spike,True,True
normal_game_trend,True,True
two_week_raw,True,True


,check,passed
0,alert_archive_2023_only,True
1,canonical_archive_2023_only,True
2,baseline_precedes_confirmation,True
3,confirmation_ends_on_alert_week,True
4,first_outcome_after_alert,True
5,second_outcome_after_first,True
6,primary_contains_no_confirmed_partial,True
7,primary_retains_suspected_rows,True
8,confirmed_evidence_after_trigger,True
9,confirmed_evidence_before_next_game,True


,field,rows_compared,mismatch_rows,maximum_absolute_difference,matched
0,future_n,660,0,0.000000e+00,True
1,next_game_value,660,0,0.000000e+00,True
2,future_week_1,660,0,0.000000e+00,True
3,future_week_2,660,0,0.000000e+00,True
4,future_mean,660,0,8.326673e-17,True
5,next_game_retention,660,0,8.881784e-16,True
6,retention,660,0,8.881784e-16,True
7,persistent,660,0,0.000000e+00,True
8,immediate_reversion,660,0,0.000000e+00,True


## Takeaways

- `rb_carry_share`: `ADVANCE_UNCHANGED_TO_FOLD_4`. This is legitimate under the frozen point gates, but fragile and not validated.
- `rb_opportunity_share`: `CONTINUE_UNCHANGED_SHADOW_FOLD_4`. Its stronger 2023 aggregate cannot waive the pre-existing 2021 direction failure.
- `wr_target_share`: `REMAIN_RETIRED`.
- `te_target_share`: `REMAIN_RETIRED`.

Required caveats: carry's lift CI includes zero; carry-only 2023 alerts underperform overlapping alerts; direction strata are not themselves equal-volume; the source files span through 2025 and are scanned before 2023 filtering, although no post-2023 row reaches scoring; the runner was not itself checkpointed before execution.